In [1]:
import pandas as pd
import numpy as np
import sys
import os

sys.path.append(os.path.abspath(".."))
from src.data_processing import BaseDataProcessor

## Data Extraction

In [2]:
src_file = r'..\data\raw\TWO_CENTURIES_OF_UM_RACES.csv'

In [3]:
um_races = BaseDataProcessor(src_file)
um_races.standardize_column_names()

2026-05-31 12:04:46 - INFO - Data loaded successfully from ..\data\raw\TWO_CENTURIES_OF_UM_RACES.csv
2026-05-31 12:04:46 - INFO - Column names standardized


In [4]:
data_info = um_races.basic_info()
data_info  

{'num_rows': '7,461,195',
 'num_cols': '13',
 'total_cells': '96,995,535',
 'missing_cells': '4,000,912',
 'missing_pct': '4.12%',
 'memory_usage': '4385.38 MB'}

In [5]:
data_details = um_races.dataset_details()
data_details

,column_name,dtype,non_null,null_count,null_pct,unique,sample_records
0,year_of_event,int64,7461195,0,0.00,146,"[2018, 2016, 2017, 2019, 2020]"
1,event_dates,object,7461195,0,0.00,14425,"[06.01.2018, 05.01.2018, 05.-07.01.2018, 05.-0..."
2,event_name,object,7461195,0,0.00,26907,"[Selva Costera (CHI), 6 Stunden Self-Transcend..."
3,event_distance/length,object,7460142,1053,0.01,2159,"[50km, 6h, 63.9km, 50mi, 28mi]"
4,event_number_of_finishers,int64,7461195,0,0.00,1453,"[22, 9, 24, 36, 34]"
5,athlete_performance,object,7461193,2,0.00,318200,"[4:51:39 h, 5:15:45 h, 5:16:44 h, 5:34:13 h, 5..."
6,athlete_club,object,4634671,2826524,37.88,552188,"[Tnfrc, Roberto Echeverría, Puro Trail Osorno,..."
7,athlete_country,object,7461192,3,0.00,208,"[CHI, ARG, COL, RUS, SLO]"
8,athlete_year_of_birth,float64,6873034,588161,7.88,193,"[1978.0, 1981.0, 1987.0, 1976.0, 1992.0]"
9,athlete_gender,object,7461188,7,0.00,3,"[M, F, X]"


### Key observations from the dataset
- The dataset uses non-standard column names with mixed casing and spaces — rename column to standardized.
- `Athlete age` and `Athlete average speed` contain nulls.
- `Athlete performance` stores finish time as a string (HH:MM:SS) — needs parsing.
- `Name of event country` may be embedded inside the event name column — needs extraction.
- `Event distance/length` mixes numeric and categorical values (e.g. `50km`, `50mi`, `6h`).

In [6]:
# drop duplicates
um_races.remove_duplicates()

2026-05-31 12:06:01 - INFO - Removed 50 duplicate rows


In [7]:
um_races.data.columns

Index(['year_of_event', 'event_dates', 'event_name', 'event_distance/length',
       'event_number_of_finishers', 'athlete_performance', 'athlete_club',
       'athlete_country', 'athlete_year_of_birth', 'athlete_gender',
       'athlete_age_category', 'athlete_average_speed', 'athlete_id'],
      dtype='object')

In [8]:
col_mapping = {
    'year_of_event'                   : 'year',
    'event_dates'                     : 'event_date_str',
    'event_distance/length'           : 'event_distance',
    'athlete_performance'             : 'athlete_finish_time_str',
    'athlete_year_of_birth'           : 'birth_year',
    'athlete_gender'                  : 'gender',
    'athlete_age_category'            : 'age_category',
    # 'athlete_average_speed'           : 'avg_speed_kmh',
    # 'event_name'                      : 'event_name',
    # 'event_number_of_finishers'       : 'num_finishers',
    # 'athlete_club'                    : 'athlete_club',
    # 'athlete_country'                 : 'country',
    # 'athlete_id'                      : 'athlete_id',
}
um_races.rename_columns(col_mapping)

2026-05-31 12:06:01 - INFO - Renamed columns: year_of_event, event_dates, event_distance/length, athlete_performance, athlete_year_of_birth, athlete_gender, athlete_age_category to year, event_date_str, event_distance, athlete_finish_time_str, birth_year, gender, age_category


In [9]:
um_races.data.sample(10)

,year,event_date_str,event_name,event_distance,event_number_of_finishers,athlete_finish_time_str,athlete_club,athlete_country,birth_year,gender,age_category,athlete_average_speed,athlete_id
1610371,2017,19.08.2017,Magurski Ultramarathon 58km (POL),58km,158,8:52:33 h,Przymierze Wojowników,POL,1976.0,M,M40,6.535,612348
4990676,2010,17.10.2010,Hengchun Airport ultra marathon (TPE),45.6km,851,4:17:37 h,NaN,TPE,1957.0,M,M50,10.62,1246112
1200201,2016,15.-16.10.2016,Raleigh Challenge Wilson Trail - Night Course ...,46.4km,124,15:00:16 h,NaN,HKG,1992.0,M,M23,3.092,318954
851850,2016,19.06.2016,Okinoshima 50km Ultra Marathon (JPN),50km,482,6:50:12 h,NaN,JPN,NaN,M,NaN,7.314,261731
1813006,2019,03.03.2019,Trail De La Grande Baume (FRA),45km,326,5:56:46 h,Marseille Running Company,POR,1982.0,M,M35,7.568,306371
6766268,2015,20.-22.11.2015,Oxfam Trailwalker Hong Kong (HKG),100km,4651,38:57:00 h,NaN,HKG,1975.0,M,M35,2.567,531041
4133190,2004,19.09.2004,Tango 100 km Ultramarathon (JPN),100km,361,12:19:49 h,NaN,JPN,1978.0,M,M23,8.11,505408
2363659,2019,24.08.2019,Tromsø Mountain Ultra (NOR),50km,81,6:57:17 h,Tromsø Triathlonklubb,NOR,1984.0,M,M23,7.189,768606
2775791,2021,20.-22.03.2021,Ohio's Backyard Ultra (USA),55h,99,67.060 km,"*Jackson, OH",USA,2002.0,M,MU23,10,863913
4041648,2003,19.10.2003,River Shimanto 100km (JPN),100km,1194,13:48:58 h,サンダーバード,JPN,1970.0,F,W23,7.238,1150740


The dataset fields can be classified and reorganized into three logical categories:

- **Event Attributes**: Information that defines and describes an event.
- **Athlete Attributes**: Information related to athlete demographics and participant details.
- **Performance Facts**: Event-specific performance metrics and results recorded for each athlete.

In [12]:
event_cols = ['year', 'event_date_str', 'event_name', 'event_distance',]
athlete_cols = ['athlete_id', 'birth_year', 'gender', 'athlete_club', 'athlete_country',]
fact_cols = ['age_category', 'event_number_of_finishers', 'athlete_finish_time_str', 'athlete_average_speed',]

In [13]:
um_races.data[event_cols + athlete_cols + fact_cols].head()

,year,event_date_str,event_name,event_distance,athlete_id,birth_year,gender,athlete_club,athlete_country,age_category,event_number_of_finishers,athlete_finish_time_str,athlete_average_speed
0,2018,06.01.2018,Selva Costera (CHI),50km,0,1978.0,M,Tnfrc,CHI,M35,22,4:51:39 h,10.286
1,2018,06.01.2018,Selva Costera (CHI),50km,1,1981.0,M,Roberto Echeverría,CHI,M35,22,5:15:45 h,9.501
2,2018,06.01.2018,Selva Costera (CHI),50km,2,1987.0,M,Puro Trail Osorno,CHI,M23,22,5:16:44 h,9.472
3,2018,06.01.2018,Selva Costera (CHI),50km,3,1976.0,M,Columbia,ARG,M40,22,5:34:13 h,8.976
4,2018,06.01.2018,Selva Costera (CHI),50km,4,1992.0,M,Baguales Trail,CHI,M23,22,5:54:14 h,8.469


In [14]:
dir_path = r'..\data\interim'
os.makedirs(dir_path, exist_ok=True)
um_races.data.to_csv(os.path.join(dir_path, 'ultra_marathon_races_cleaned.csv'), index=False)

The dataset has been standardized by applying consistent column naming conventions, reorganizing fields into a logical structure, and removing duplicate records. With these foundational cleaning steps completed, the dataset is now ready for further refinement and analysis.
